# 04 — PaySim LightGBM candidate review

This is a **review-first** notebook for the E1–E4 model-family spike. The preferred execution path is `make model-spike` or `.\make.ps1 model-spike`; this notebook reads the resulting machine-readable manifest and explains what the comparisons mean.

It does not lock the final model or FeatureSpec, and it does not claim production fraud performance.

In [1]:
from pathlib import Path

from IPython.display import Markdown, display

from pit_fintech.data.paysim import (
    find_paysim_csv,
    resolve_project_root,
    setup_instructions,
)
from pit_fintech.models.paysim_lightgbm import (
    DEFAULT_FIXED_FPR,
    DEFAULT_NONFRAUD_SAMPLE_PER_GROUP,
    DEFAULT_SEED,
    find_latest_candidate_manifest,
    load_candidate_manifest,
    manifest_summary_rows,
    run_paysim_lightgbm_spike,
)

PROJECT_ROOT = resolve_project_root(Path.cwd())
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
paysim_csv = find_paysim_csv(PROJECT_ROOT)

## 1. Explicit execution control

Keep `RUN_TRAINING = False` for normal use and notebook verification. Set it to `True` only after launching `lab-training` when you intentionally want a new MLflow parent run.

LightGBM feature-name messages and MLflow's unpinned-pip environment message are non-blocking warnings. Review mode avoids retraining and therefore avoids repeating them.

In [2]:
RUN_TRAINING = False
NONFRAUD_SAMPLE_PER_GROUP = DEFAULT_NONFRAUD_SAMPLE_PER_GROUP
SEED = DEFAULT_SEED
FIXED_FPR = DEFAULT_FIXED_FPR

execution_config = {
    "mode": "train-and-review" if RUN_TRAINING else "review-only",
    "nonfraud_sample_per_group": NONFRAUD_SAMPLE_PER_GROUP,
    "seed": SEED,
    "validation_fixed_fpr": FIXED_FPR,
    "paysim_csv_found": paysim_csv is not None,
}
execution_config

{'mode': 'train-and-review',
 'nonfraud_sample_per_group': 5000,
 'seed': 20260727,
 'validation_fixed_fpr': 0.01,
 'paysim_csv_found': True}

In [3]:
run_result = None
if not RUN_TRAINING:
    print("Review-only mode: no model will be trained from this notebook.")
elif paysim_csv is None:
    display(Markdown("```text\n" + setup_instructions(PROJECT_ROOT) + "\n```"))
else:
    run_result = run_paysim_lightgbm_spike(
        paysim_csv,
        project_root=PROJECT_ROOT,
        artifact_root=ARTIFACT_ROOT,
        nonfraud_sample_per_group=NONFRAUD_SAMPLE_PER_GROUP,
        seed=SEED,
        fixed_fpr=FIXED_FPR,
    )
    print("new manifest:", run_result[1])

C:\workspace\pit-fintech\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\workspace\pit-fintech\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\workspace\pit-fintech\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/27 11:53:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
C:\workspace\pit-fintech\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\workspace\pit-fintech\.v

new manifest: C:\workspace\pit-fintech\artifacts\experiments\paysim-lightgbm-spike\32f0200fc35044668ef3c2774f58864a\manifest.json


## 2. Load the latest validated evidence

The same manifest is produced whether the spike was launched from Make, PowerShell, or this notebook. Loading it through the Pydantic contract checks the expected schema before interpretation.

In [4]:
manifest_path = run_result[1] if run_result else find_latest_candidate_manifest(ARTIFACT_ROOT)
manifest = load_candidate_manifest(manifest_path) if manifest_path else None

if manifest is None:
    display(Markdown("No candidate manifest found. Run `.\\make.ps1 model-spike` first."))
else:
    print("manifest:", manifest_path)
    print("status / parent run:", manifest.status, manifest.mlflow_parent_run_id)
    print("dataset snapshot:", manifest.dataset_snapshot_id)
    print("cohort rows / fraud:", manifest.cohort_rows, manifest.cohort_fraud_rows)
    display(manifest_summary_rows(manifest))

manifest: C:\workspace\pit-fintech\artifacts\experiments\paysim-lightgbm-spike\32f0200fc35044668ef3c2774f58864a\manifest.json
status / parent run: completed 32f0200fc35044668ef3c2774f58864a
dataset snapshot: paysim1:16910f90577b0d98
cohort rows / fraud: 38213 8213


[{'experiment': 'E1',
  'features': 'static',
  'split': 'temporal',
  'test_pr_auc': 0.176951,
  'test_roc_auc': 0.627406,
  'recall_at_fixed_fpr': 0.0,
  'observed_fpr': 0.0,
  'threshold_policy': 'zero-positive-fallback',
  'seconds': 0.113,
  'mlflow_run_id': 'ea28ef220d0a4e3583a1da6aa2508f8f'},
 {'experiment': 'E2',
  'features': 'leaky',
  'split': 'random',
  'test_pr_auc': 0.915528,
  'test_roc_auc': 0.967541,
  'recall_at_fixed_fpr': 0.677223,
  'observed_fpr': 0.013831,
  'threshold_policy': 'max-tpr-within-fpr',
  'seconds': 1.216,
  'mlflow_run_id': '350d1b547aa140b6b191d5db4b5f1835'},
 {'experiment': 'E3',
  'features': 'pit',
  'split': 'random',
  'test_pr_auc': 0.892585,
  'test_roc_auc': 0.953843,
  'recall_at_fixed_fpr': 0.622412,
  'observed_fpr': 0.012165,
  'threshold_policy': 'max-tpr-within-fpr',
  'seconds': 0.896,
  'mlflow_run_id': 'a5100e650cd14196ac7bfc6134c2b25d'},
 {'experiment': 'E4',
  'features': 'pit',
  'split': 'temporal',
  'test_pr_auc': 0.324524,


## 3. What the E1–E4 matrix tests

| ID | Features | Split | Question |
|---|---|---|---|
| E1 | Static request-time | Temporal | What can the model do without history? |
| E2 | Current/future/lifetime controls | Random | How optimistic can invalid leakage become? |
| E3 | Strict-PIT recipient history | Random | How optimistic is random splitting with safe features? |
| E4 | Strict-PIT recipient history | Temporal | What is the closest candidate to deployment semantics? |

`E4 − E1` estimates the value of PIT history under the same temporal split. `E3 − E4` exposes split-policy optimism. `E2 − E3` exposes the additional optimism from deliberately leaky features.

In [5]:
comparisons = None
if manifest is not None:
    results = {result.experiment_id: result for result in manifest.experiments}
    comparisons = {
        "pit_history_value__E4_minus_E1_pr_auc": round(
            results["E4"].test_pr_auc - results["E1"].test_pr_auc,
            6,
        ),
        "random_split_optimism__E3_minus_E4_pr_auc": round(
            results["E3"].test_pr_auc - results["E4"].test_pr_auc,
            6,
        ),
        "additional_leakage__E2_minus_E3_pr_auc": round(
            results["E2"].test_pr_auc - results["E3"].test_pr_auc,
            6,
        ),
        "main_candidate__E4_recall_at_fixed_fpr": round(
            results["E4"].test_recall_at_fixed_fpr,
            6,
        ),
        "main_candidate__E4_observed_test_fpr": round(
            results["E4"].test_observed_fpr,
            6,
        ),
    }
    display(comparisons)

{'pit_history_value__E4_minus_E1_pr_auc': 0.147573,
 'random_split_optimism__E3_minus_E4_pr_auc': 0.568061,
 'additional_leakage__E2_minus_E3_pr_auc': 0.022943,
 'main_candidate__E4_recall_at_fixed_fpr': 0.202875,
 'main_candidate__E4_observed_test_fpr': 0.0015}

## 4. Reproducibility and resource evidence

Model metrics are useful only when tied to the exact dataset, code, dependency lock, feature list and split membership. The following cell exposes those identifiers and the lightweight CPU/RSS measurements recorded for every experiment.

In [6]:
if manifest is not None:
    lineage = {
        "dataset_snapshot_id": manifest.dataset_snapshot_id,
        "candidate_table_checksum": manifest.candidate_table_checksum,
        "code_commit": manifest.code_commit,
        "dependency_lock_sha256": manifest.dependency_lock_sha256,
        "candidate_feature_version": manifest.candidate_feature_version,
        "mlflow_parent_run_id": manifest.mlflow_parent_run_id,
    }
    resources = [
        {
            "experiment": result.experiment_id,
            "best_iteration": result.best_iteration,
            "training_seconds": round(result.training_seconds, 3),
            "rss_delta_mib": round(
                (result.process_rss_after_bytes - result.process_rss_before_bytes) / (1024**2),
                3,
            ),
            "training_dataset_checksum": result.training_dataset_checksum,
            "model_uri": result.model_uri,
        }
        for result in manifest.experiments
    ]
    display(lineage)
    display(resources)

{'dataset_snapshot_id': 'paysim1:16910f90577b0d98',
 'candidate_table_checksum': '530167952381586c118a5c39aa0539bc3ee6af9d1c13f869a7140ed5117dfde6',
 'code_commit': '115e98dcf10b3071f02c228925ab0b366dd2e611-dirty',
 'dependency_lock_sha256': '4200c93b6037ee9f252b810016b82961f0abb70b5502348521f92f542f3c3cf4',
 'candidate_feature_version': 'paysim-recipient-v1',
 'mlflow_parent_run_id': '32f0200fc35044668ef3c2774f58864a'}

[{'experiment': 'E1',
  'best_iteration': 1,
  'training_seconds': 0.113,
  'rss_delta_mib': 2.012,
  'training_dataset_checksum': '5b3e1476c8b2ee82ee0ae10674e29073219a9a2580fd3e7f9a42da28d12e0d29',
  'model_uri': 'runs:/ea28ef220d0a4e3583a1da6aa2508f8f/model'},
 {'experiment': 'E2',
  'best_iteration': 250,
  'training_seconds': 1.216,
  'rss_delta_mib': 6.508,
  'training_dataset_checksum': 'a77173849f34ba228e2845f64120373f7f30ce5a66a33d36db385af51bc00101',
  'model_uri': 'runs:/350d1b547aa140b6b191d5db4b5f1835/model'},
 {'experiment': 'E3',
  'best_iteration': 263,
  'training_seconds': 0.896,
  'rss_delta_mib': 0.172,
  'training_dataset_checksum': '9f2fcdef346d623061d77f9de98244433427fa0642a1d45aabc25fafbc393969',
  'model_uri': 'runs:/a5100e650cd14196ac7bfc6134c2b25d/model'},
 {'experiment': 'E4',
  'best_iteration': 1,
  'training_seconds': 0.161,
  'rss_delta_mib': -0.945,
  'training_dataset_checksum': '6bc0b2671e06a9a70679b18226df5ad4f095d26cd92666856d5bf249a40fecbb',
  'mode

## 5. Decision boundary

- E2 is deliberately invalid and can never be promoted.
- E3 is useful for diagnosing random-split optimism, not deployment performance.
- E4 is the main candidate, but the sampled cohort does not represent natural PaySim prevalence.
- A weak temporal result must not be hidden through tuning; it is valid evidence about drift and feature utility.
- Freeze FeatureSpec v1 only after reviewing E1–E4, cold-start behavior, lineage and resource cost.
- The next locked baseline must read from PaySim Silver and run from a clean commit.

In [7]:
if manifest is not None:
    print("Claim boundaries from the manifest:")
    for statement in manifest.claim_boundary:
        print("-", statement)

Claim boundaries from the manifest:
- The cohort oversamples fraud, so absolute PR-AUC is not a production prevalence estimate.
- E2 is a deliberately leaky positive control and can never be promoted.
- LightGBM remains a candidate until the user reviews this evidence and freezes FeatureSpec v1.
- This spike does not implement Feast, Gold backfill, Redis parity, promotion, or serving.
